# Jasmine v4 — LAMA Cleanup (P0-39)
Removes Navier-Stokes brown/orange square artifact spots from all 62 training images.

**Runtime:** GPU (T4). Go to Runtime → Change runtime type → T4 GPU before running.

In [ ]:
# CELL 1 — Install
!pip install simple-lama-inpainting opencv-python-headless -q
print('Install done')

In [ ]:
# CELL 2 — Download zip from Google Drive (avoids browser upload timeout)
import os, zipfile

file_id = "1uMqF10U7NDduOlQub6bOsM1jPa7vglpo"
!gdown --id {file_id} -O /content/jasmine_v4_clean_62.zip

os.makedirs('/content/v4_clean', exist_ok=True)
os.makedirs('/content/v4_lama', exist_ok=True)

with zipfile.ZipFile('/content/jasmine_v4_clean_62.zip', 'r') as z:
    z.extractall('/content/v4_clean')

images = sorted([f for f in os.listdir('/content/v4_clean') if f.endswith('.png')])
print(f'{len(images)} images extracted')

In [ ]:
# CELL 3 — Preview 3 originals before cleanup
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

preview_files = [images[0], images[len(images)//2], images[-1]]

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
for ax, fname in zip(axes, preview_files):
    img = cv2.imread(f'/content/v4_clean/{fname}')
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(f'BEFORE: {fname[:30]}', fontsize=8)
    ax.axis('off')
plt.suptitle('BEFORE — 3 sample originals', fontsize=14)
plt.tight_layout()
plt.show()
print('Review spots visible on shoulders/chest before proceeding to cleanup')

In [ ]:
# CELL 4 — Watermark removal (watermark-only mask, v3 — 35% width × 10% height)
from simple_lama_inpainting import SimpleLama
import numpy as np
import cv2
from PIL import Image
import os

lama = SimpleLama()

def make_watermark_mask(img_bgr):
    h, w = img_bgr.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    mask[h - int(h * 0.10):, w - int(w * 0.35):] = 255
    return mask

for i, fname in enumerate(images):
    src = f'/content/v4_clean/{fname}'
    dst = f'/content/v4_lama/{fname}'
    img_bgr = cv2.imread(src)
    mask = make_watermark_mask(img_bgr)
    result = lama(Image.open(src).convert('RGB'), Image.fromarray(mask))
    result.save(dst)
    print(f'[{i+1:02d}/62] {fname[:40]} → done')

print('\nDone. All 62 images cleaned.')

In [ ]:
# CELL 5 — Before/After comparison (first, middle, last image)
import matplotlib.pyplot as plt
import cv2

picks = [images[0], images[30], images[61]]

fig, axes = plt.subplots(3, 2, figsize=(16, 28))
for row, fname in enumerate(picks):
    before = cv2.cvtColor(cv2.imread(f'/content/v4_clean/{fname}'), cv2.COLOR_BGR2RGB)
    after  = cv2.cvtColor(cv2.imread(f'/content/v4_lama/{fname}'), cv2.COLOR_BGR2RGB)
    axes[row, 0].imshow(before)
    axes[row, 0].set_title(f'BEFORE: {fname[:30]}', fontsize=8)
    axes[row, 0].axis('off')
    axes[row, 1].imshow(after)
    axes[row, 1].set_title('AFTER: watermark removed', fontsize=8)
    axes[row, 1].axis('off')

plt.tight_layout()
plt.savefig('/content/before_after_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 6 — Download cleaned images as zip
import zipfile
import os
from google.colab import files

out_zip = '/content/jasmine_v4_lama_cleaned.zip'
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir('/content/v4_lama'):
        if fname.endswith('.png'):
            zf.write(f'/content/v4_lama/{fname}', fname)

size_mb = os.path.getsize(out_zip) / 1024 / 1024
print(f'Zip ready: {size_mb:.0f}MB — downloading now...')
files.download(out_zip)
print('DONE. Save as: ~/Desktop/Instagram/03_ai_models/jasmine_mako/02_training_data/v4_clean_lama/')